In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv


In [2]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, r2_score

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import xgboost as xgb

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [5]:
train=pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv")
test=pd.read_csv("/kaggle/input/playground-series-s6e1/test.csv")
sample=pd.read_csv("/kaggle/input/playground-series-s6e1/sample_submission.csv")

In [6]:
train.head(5)

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [7]:
test.head(5)

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate


In [8]:
sample.head()

,id,exam_score
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [9]:
train.isnull().sum()

id                  0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64

In [10]:
train.columns

Index(['id', 'age', 'gender', 'course', 'study_hours', 'class_attendance',
       'internet_access', 'sleep_hours', 'sleep_quality', 'study_method',
       'facility_rating', 'exam_difficulty', 'exam_score'],
      dtype='object')

In [11]:
train.dtypes

id                    int64
age                   int64
gender               object
course               object
study_hours         float64
class_attendance    float64
internet_access      object
sleep_hours         float64
sleep_quality        object
study_method         object
facility_rating      object
exam_difficulty      object
exam_score          float64
dtype: object

In [12]:
train.shape

(630000, 13)

In [13]:
train.drop("id", axis=1, inplace=True)
test.drop("id", axis=1, inplace=True)

In [14]:
numerical_cols = [
    "age",
    "study_hours",
    "class_attendance",
    "sleep_hours"
]

categorical_cols = [
    "gender",
    "course",
    "internet_access",
    "sleep_quality",
    "study_method",
    "facility_rating",
    "exam_difficulty"
]

In [15]:
X = train.drop("exam_score", axis=1)
y = train["exam_score"]

In [16]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# trainin CatBoost Model

In [17]:
cat_features_idx = [X.columns.get_loc(col) for col in categorical_cols]

In [18]:
cat_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=200
)

In [19]:
cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_features_idx,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 18.5494898	test: 18.4786419	best: 18.4786419 (0)	total: 831ms	remaining: 27m 41s
200:	learn: 8.8375220	test: 8.8347330	best: 8.8347330 (200)	total: 1m 34s	remaining: 14m 2s
400:	learn: 8.8148632	test: 8.8190303	best: 8.8190303 (400)	total: 2m 43s	remaining: 10m 51s
600:	learn: 8.7878369	test: 8.7999401	best: 8.7999401 (600)	total: 4m 11s	remaining: 9m 44s
800:	learn: 8.7679767	test: 8.7878314	best: 8.7878314 (800)	total: 5m 39s	remaining: 8m 28s
1000:	learn: 8.7509245	test: 8.7786573	best: 8.7786573 (1000)	total: 7m 5s	remaining: 7m 4s
1200:	learn: 8.7359407	test: 8.7722090	best: 8.7722090 (1200)	total: 8m 31s	remaining: 5m 39s
1400:	learn: 8.7216462	test: 8.7664585	best: 8.7664585 (1400)	total: 9m 57s	remaining: 4m 15s
1600:	learn: 8.7080043	test: 8.7616662	best: 8.7616662 (1600)	total: 11m 24s	remaining: 2m 50s
1800:	learn: 8.6953257	test: 8.7580826	best: 8.7580826 (1800)	total: 12m 51s	remaining: 1m 25s
1999:	learn: 8.6819151	test: 8.7540587	best: 8.7540572 (1998)	total: 1

In [20]:
pred_cat=cat_model.predict(X_val)

In [21]:
rmse_cat=root_mean_squared_error(y_val, pred_cat)
r2_cat=r2_score(y_val, pred_cat)
print("CatBoost RMSE: ", rmse_cat)
print("CatBoost R2: ", r2_cat)

CatBoost RMSE:  8.754057161002427
CatBoost R2:  0.7845237461671193


In [22]:
test_pred_cat=cat_model.predict(test)
sample["exam_score"]=test_pred_cat
sample.to_csv("cat_submission.csv", index=False)
print("Successfully Saved CatBoost Submission")

Successfully Saved CatBoost Submission


# Training XGBoost Model

In [23]:
preprocessor=ColumnTransformer(
    [
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

In [24]:
xgb_model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

In [25]:
xgb_pipeline = Pipeline(
    [
        ("prep", preprocessor),
        ("model", xgb_model)
    ]
)

In [26]:
xgb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['gender', 'course',
                                                   'internet_access',
                                                   'sleep_quality',
                                                   'study_method',
                                                   'facility_rating',
                                                   'exam_difficulty']),
                                                 ('num', 'passthrough',
                                                  ['age', 'study_hours',
                                                   'class_attendance',
                                                   'sleep_hours'])])),
                ('model',
                 XGBRegressor(base_score=...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=7, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=2000, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [27]:
pred_xgb=xgb_pipeline.predict(X_val)

In [28]:
rmse_xgb = root_mean_squared_error(y_val, pred_xgb)
r2_xgb = r2_score(y_val, pred_xgb)

print("XGBoost RMSE:", rmse_xgb)
print("XGBoost R2:", r2_xgb)

XGBoost RMSE: 8.74140915455817
XGBoost R2: 0.7851459436659922


In [29]:
test_pred_xgb=xgb_pipeline.predict(test)
sample["exam_score"]=test_pred_xgb
sample.to_csv("xgb_submission.csv", index=False)
print("Successfully Saved XGBoost Submission")

Successfully Saved XGBoost Submission


# Training LGBMRegressor

In [30]:
lgb_model = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=8,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [31]:
lgb_pipeline = Pipeline(
    [
        ("prep", preprocessor),
        ("model", lgb_model)
    ]
)

In [32]:
lgb_pipeline.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012270 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 595
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 11
[LightGBM] [Info] Start training from score 62.482335


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['gender', 'course',
                                                   'internet_access',
                                                   'sleep_quality',
                                                   'study_method',
                                                   'facility_rating',
                                                   'exam_difficulty']),
                                                 ('num', 'passthrough',
                                                  ['age', 'study_hours',
                                                   'class_attendance',
                                                   'sleep_hours'])])),
                ('model',
                 LGBMRegressor(colsample_bytree=0.8, learning_rate=0.02,
                               max_depth=8, n_estimators=3000, num_leaves=64,
                               random_state=42, subsample=0.8))])

In [33]:
lgb_pred=lgb_pipeline.predict(X_val)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [34]:
rmse_lgb = root_mean_squared_error(y_val, lgb_pred)
r2_lgb = r2_score(y_val, lgb_pred)

print("LightGBM RMSE:", rmse_lgb)
print("LightGBM R2:", r2_lgb)

LightGBM RMSE: 8.748119862369034
LightGBM R2: 0.7848159336627363


In [35]:
test_pred_lgb=lgb_pipeline.predict(test)
sample["exam_score"]=test_pred_lgb
sample.to_csv("lgb_submission.csv", index=False)
print("Successfully Saved LGBoost Submission")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Successfully Saved LGBoost Submission


# let's Ensemble all

In [36]:
pred_ensemble = (
    0.3 * pred_cat +
    0.4 * pred_xgb +
    0.3 * lgb_pred
)

rmse_ens = root_mean_squared_error(y_val, pred_ensemble)
r2_ens = r2_score(y_val, pred_ensemble)

print("ENSEMBLE RMSE:", rmse_ens)
print("ENSEMBLE R2:", r2_ens)


ENSEMBLE RMSE: 8.727932813191858
ENSEMBLE R2: 0.7858078997988778


In [37]:
test_preds = (
    0.3 * cat_model.predict(test) +
    0.4 * xgb_pipeline.predict(test) +
    0.3 * lgb_pipeline.predict(test)
)

sample["exam_score"] = test_preds
sample.to_csv("submission.csv", index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
